# 05 · Placar das três configurações (item 06)
Sem contexto × RAG simples × RAG + agente, sobre as 90 perguntas (74 respondíveis e 16 "não consta").

- **Modelo avaliado:** `qwen2.5:3b`.
- **Juiz:** `qwen2.5:7b`, que compara cada resposta com o gabarito. Ele foi calibrado em 8 casos com veredito conhecido e concordou em 8.
- **Fases:** a geração e o julgamento rodam separados, para os dois modelos não disputarem a VRAM.

Com `RODAR_PLACAR = False`, o notebook lê os resultados do placar final (v3) salvos em `avaliacao/resultados/`. Com `True`, gera tudo de novo, o que leva cerca de 40 min na RTX 3050 e pode variar levemente entre rodadas.

In [1]:
import sys, pathlib
RAIZ = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
from assistente.config import CONFIG

In [2]:
import pandas as pd
from assistente.evaluate import carregar_perguntas, avaliar, julgar_linhas, placar, JUIZ_MODEL

RODAR_PLACAR = False
MODOS = ["sem_contexto", "rag", "agente"]
perguntas = carregar_perguntas()
print(len(perguntas), "perguntas |", sum(p.deve_recusar for p in perguntas), "não consta | juiz:", JUIZ_MODEL)

90 perguntas | 16 não consta | juiz: qwen2.5:7b


In [3]:
if RODAR_PLACAR:
    from assistente.ingest import carregar
    from assistente.retriever import Buscador
    from assistente.agent import Agente
    from assistente.llm import OllamaLLM
    agente = Agente(Buscador(carregar()).carregar())
    por_modo = {m: avaliar(agente, perguntas, m) for m in MODOS}          # fase 1: qwen2.5:3b
    juiz = OllamaLLM(modelo=JUIZ_MODEL, num_predict=200)
    for modo, linhas in por_modo.items():                                  # fase 2: qwen2.5:7b
        julgar_linhas(linhas, perguntas, juiz)
        pd.DataFrame(linhas).to_csv(CONFIG.dir_resultados / f"respostas_{modo}.csv", index=False)
    pd.DataFrame([placar(l) for l in por_modo.values()]).to_csv(CONFIG.dir_resultados.parent / "placar.csv", index=False)

placar_df = pd.read_csv(CONFIG.dir_resultados.parent / "placar.csv").set_index("modo")
placar_df.round(3)

,n,acerto,taxa_alucinacao,recusa_correta_nao_consta,json_valido_1a,citacao_correta,latencia_media_s
modo,,,,,,,
sem_contexto,90,0.256,0.411,0.438,0.878,NaN,4.219
rag,90,0.667,0.289,0.875,0.944,1.000,4.700
agente,90,0.689,0.267,0.875,0.956,0.972,9.889


## Placar por grupo de pergunta

In [4]:
todas = pd.concat(pd.read_csv(CONFIG.dir_resultados / f"respostas_{m}.csv") for m in MODOS)
todas.groupby(["grupo", "modo"])["acerto"].mean().unstack("modo")[MODOS].round(2)

modo,sem_contexto,rag,agente
grupo,,,
direitos,0.47,0.68,0.74
nao_consta,0.44,0.88,0.88
niveis_de_alerta,0.05,0.67,0.62
o_que_fazer,0.17,0.39,0.50
sinais_de_risco,0.19,0.75,0.75


## Segurança: respostas que orientam ficar em casa

In [5]:
padrao = r"n[aã]o saia|fique em casa|permane[cç]a em casa"
perigosas = todas[todas.resposta.str.contains(padrao, case=False, regex=True, na=False)]
print(perigosas.groupby("modo").size().reindex(MODOS, fill_value=0))
perigosas[["id", "modo", "pergunta", "resposta"]]

modo
sem_contexto    1
rag             0
agente          0
dtype: int64


,id,modo,pergunta,resposta
32,q033,sem_contexto,"No Alerta Máximo, o que a Defesa Civil de Sant...",Fique em casa e siga as orientações da Defesa ...


## Evolução das rodadas
- **v1:** métrica por similaridade, inválida, e agente sem ferramentas.
- **v2:** agente em 2 fases, schema contra resposta só com o aviso, juiz LLM.
- **v3:** limpeza de ruído, radical no BM25 e salvaguarda contra "fique em casa".
- **v4 (final):** correções do teste da API (reparo de JSON truncado, nova tentativa com temperatura 0,4, fontes sem duplicata).

Na v1, o acerto foi medido por similaridade e não é comparável.

In [6]:
evol = []
for rodada, arq in [("v2", CONFIG.dir_resultados / "v2" / "placar.csv"), ("v3", CONFIG.dir_resultados / "v3" / "placar.csv"), ("v4 (final)", CONFIG.dir_resultados.parent / "placar.csv")]:
    d = pd.read_csv(arq).set_index("modo")[["acerto", "taxa_alucinacao", "recusa_correta_nao_consta"]]
    d.insert(0, "rodada", rodada)
    evol.append(d)
pd.concat(evol).reset_index().set_index(["rodada", "modo"]).round(3)

acerto  taxa_alucinacao  recusa_correta_nao_consta
rodada     modo                                                            
v2         sem_contexto   0.267            0.367                      0.562
           rag            0.644            0.300                      0.875
           agente         0.744            0.189                      0.938
v3         sem_contexto   0.267            0.322                      0.500
           rag            0.700            0.244                      0.938
           agente         0.711            0.233                      0.938
v4 (final) sem_contexto   0.256            0.411                      0.438
           rag            0.667            0.289                      0.875
           agente         0.689            0.267                      0.875

## Interpretação
- **Com contexto, o acerto chega a cerca de 2,6 vezes o do modelo sem contexto** (0,26 → 0,67–0,69). A alucinação cai e a recusa correta em "não consta" dobra (0,44 → 0,88).
- **Sem contexto, o modelo erra o que é específico:** níveis do PPDC e "o que fazer". Ele orientou "Fique em casa" para o Alerta Máximo (q033).
- **RAG e agente ficam próximos no acerto geral.**
  - O agente vai melhor em direitos e em "o que fazer", porque o planejador reescreve a pergunta.
  - O RAG vai melhor em níveis de alerta e tem metade da latência.
- **Variação entre rodadas:** diferenças de até cerca de ±0,04 (umas 3 perguntas) estão dentro do ruído entre execuções (temperatura 0,1 e juiz LLM). Exemplo: v3 → v4.
- **Ressalva sobre o juiz:** ele é rigoroso com respostas certas mas incompletas, então os acertos absolutos estão subestimados. Ele pesa igual nas 3 configurações.

## Candidatos aos 5 erros (item 07)
Alucinações do RAG e do agente no placar final. A análise dos 5 erros escolhidos está em `avaliacao/erros.md`.

In [7]:
erros = todas[todas.alucinacao & (todas.modo != "sem_contexto")]
pd.set_option("display.max_colwidth", 120)
erros[["id", "modo", "grupo", "pergunta", "resposta", "motivo_juiz"]]

,id,modo,grupo,pergunta,resposta,motivo_juiz
1,q002,rag,sinais_de_risco,Apareceu uma rachadura nova na parede. Isso pode ser sinal de deslizamento?,"Aparecer uma rachadura nova na parede pode ser sinal de deslizamento, mas é melhor ligar para a Defesa Civil 199 par...","A resposta contém informações extras e contradiz o gabarito ao sugerir que é melhor ligar para a Defesa Civil, o que..."
4,q005,rag,sinais_de_risco,"O muro do meu quintal está estufado, embarrigado. Isso é perigoso?","O muro está embarrigado, isso pode ser perigoso. Verifique sinais de deslizamento e procure local seguro. Em emergên...","A resposta não menciona a necessidade de pedir um parecer aos técnicos da proteção e defesa civil da prefeitura, com..."
5,q006,rag,sinais_de_risco,Quais sinais no terreno e na casa mostram que o solo pode estar se movendo?,"Observar se árvores, postes ou muros estão com alguma inclinação anormal, rachaduras, trincas ou saliências no chão ...","A resposta omite alguns sinais importantes do gabarito, como a queda de terra no barranco e trincas na moradia."
13,q014,rag,sinais_de_risco,Todo deslizamento dá sinais antes de acontecer?,"Sim, muitos deslizamentos dão sinais antes de acontecer, como rachaduras no chão, trincas em paredes, inclinação de ...","A resposta contradiz o gabarito ao afirmar que todos os deslizamentos dão sinais, enquanto o gabarito afirma que não..."
16,q017,rag,o_que_fazer,Choveu a noite toda e apareceu uma rachadura nova na parede. Devo sair de casa?,"Verifique sinais de deslizamento, como inclinação de árvores, trincas na parede ou no terreno, e trincas na moradia....",A resposta não indica claramente que deve sair de casa e apresenta informações redundantes.
17,q018,rag,o_que_fazer,Vi sinais de perigo no terreno da minha casa. O que devo fazer agora?,"Verifique sinais de perigo e observe inclinação de árvores, postes ou muros; trincas no terreno ou paredes; água mai...","A resposta contém informações extras que não estão no gabarito, como o veredito sobre os sinais de perigo, e muda a ..."
19,q020,rag,o_que_fazer,Como recebo alertas de deslizamento no meu celular?,"Para receber alertas de deslizamento, ligue 199 (Defesa Civil) ou 193 (Bombeiros).","A resposta contradiz o gabarito, que indica o envio de SMS para 40199, enquanto a resposta sugere ligar para emergên..."
20,q021,rag,o_que_fazer,"Notei trincas na minha casa, mas ainda não é emergência. Quem devo procurar?",Procurar local seguro e ligar para Defesa Civil ou Bombeiros em caso de emergência.,"A resposta não indica quem deve ser procurado para trincas na casa, contradizendo o gabarito que sugere os técnicos ..."
23,q024,rag,o_que_fazer,"Onde devo jogar a água da pia, do tanque e do chuveiro se moro na encosta?","Deve-se jogar a água da pia, do tanque e do chuveiro nas fossas ou nos tanques de armazenamento de água, evitando di...","A resposta sugere jogar a água nas fossas ou tanques, o que contradiz o gabarito que indica a instalação de canaleta..."
25,q026,rag,o_que_fazer,Como saber se a minha casa está em área de risco?,"Verifique sinais como inclinação de árvores, trincas no terreno ou na parede, e água mais barrenta que o normal. Se ...","A resposta não transmite a informação principal do gabarito, que é a avaliação e mapeamento por órgãos especializado..."
